In [ ]:
!pip install numpy pandas matplotlib scikit-learn

# Week 4: NumPy, Centroid & KNN Classifiers, and Naive Bayes

In this notebook we'll:
1. Warm up with some NumPy basics
2. Load a **continuous-feature** dataset (Iris) and a **categorical-feature** dataset (Play Tennis)
3. Build a **Nearest-Centroid classifier** and a **K-Nearest-Neighbors classifier** from scratch on the Iris data, and check our work against scikit-learn
4. Build a **Naive Bayes classifier** from scratch on the Play Tennis data, and check our work against scikit-learn

Cells marked **&#9997;&#65039; Your turn** are for you to fill in — look for `pass` and replace it with working code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestCentroid, KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score

np.random.seed(42)
%matplotlib inline

## 1. NumPy Basics

NumPy is the core numerical library in the Python data-science stack — pandas, scikit-learn, and nearly everything else builds on top of `numpy.ndarray`. Let's warm up.

### 1.1 Creating arrays & basic operations

In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print("a:", a)
print("shape:", a.shape, "dtype:", a.dtype)

print("\nIndexing a[1:3]:", a[1:3])
print("Elementwise a + b:", a + b)
print("Elementwise a * 2:", a * 2)

# Broadcasting: a 2D array plus a 1D array applies the 1D array to every row
matrix = np.array([[1, 2, 3],
                    [4, 5, 6]])
print("\nmatrix + [10, 20, 30]:\n", matrix + np.array([10, 20, 30]))

### 1.2 Aggregations

`mean`, `std`, and `sum` can act over the whole array or along a specific `axis`. For a 2D array, `axis=0` collapses rows (one value per column) and `axis=1` collapses columns (one value per row).

In [ ]:
scores = np.array([[90, 85, 78],
                    [70, 88, 95],
                    [60, 75, 82]])

print("Overall mean:", scores.mean())
print("Mean per column (axis=0):", scores.mean(axis=0))
print("Mean per row (axis=1):", scores.mean(axis=1))
print("Std per column (axis=0):", scores.std(axis=0))

### 1.3 &#9997;&#65039; Your turn: Euclidean distance

Both the centroid classifier and the KNN classifier below need a way to measure "how far apart" two feature vectors are. The most common choice is **Euclidean distance**:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_i (a_i - b_i)^2}$$

Complete `euclidean_distance` below using NumPy (no Python `for` loop needed — use vectorized operations).

In [ ]:
def euclidean_distance(a, b):
    """Euclidean distance between two 1D numpy arrays."""
    pass

In [ ]:
# Sanity check
assert np.isclose(euclidean_distance(np.array([0, 0]), np.array([3, 4])), 5.0)
assert np.isclose(euclidean_distance(np.array([1, 1, 1]), np.array([1, 1, 1])), 0.0)
print("Passed.")

## 2. The Datasets

### 2.1 A continuous-feature dataset: Iris

The classic [Iris dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-plants-dataset) has 4 continuous features (measured in cm) and 3 classes of flower. It ships with scikit-learn, so no download is needed.

In [ ]:
iris = datasets.load_iris()
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(iris_df.shape)
iris_df.head()

In [ ]:
iris_df.describe()

In [ ]:
plt.figure(figsize=(6, 5))
for species in iris.target_names:
    subset = iris_df[iris_df["species"] == species]
    plt.scatter(subset["petal length (cm)"], subset["petal width (cm)"], label=species)
plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.title("Iris: petal length vs. petal width")
plt.legend()
plt.show()

### 2.2 A categorical-feature dataset: Play Tennis

This is the classic teaching dataset for Naive Bayes: given the day's weather conditions, was tennis played? Every feature here is categorical — there's nothing to average.

In [ ]:
play_tennis = pd.DataFrame({
    "Outlook":     ["Sunny", "Sunny", "Overcast", "Rain", "Rain", "Rain", "Overcast",
                     "Sunny", "Sunny", "Rain", "Sunny", "Overcast", "Overcast", "Rain"],
    "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool", "Cool", "Cool",
                     "Mild", "Cool", "Mild", "Mild", "Mild", "Hot", "Mild"],
    "Humidity":    ["High", "High", "High", "High", "Normal", "Normal", "Normal",
                     "High", "Normal", "Normal", "Normal", "High", "Normal", "High"],
    "Wind":        ["Weak", "Strong", "Weak", "Weak", "Weak", "Strong", "Strong",
                     "Weak", "Weak", "Weak", "Strong", "Strong", "Weak", "Strong"],
    "PlayTennis":  ["No", "No", "Yes", "Yes", "Yes", "No", "Yes",
                     "No", "Yes", "Yes", "Yes", "Yes", "Yes", "No"],
})

play_tennis

In [ ]:
for col in play_tennis.columns:
    print(f"\n{col}:")
    print(play_tennis[col].value_counts())

### 2.3 Train/test split (Iris)

We'll hold out 30% of the Iris data as a test set, stratified by species so each class is proportionally represented in both splits.

In [ ]:
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 3. Nearest-Centroid Classifier (Iris)

**Idea:** for each class, compute the *centroid* — the average feature vector of all training examples in that class. To classify a new point, measure its distance to every centroid and pick the closest one.

It's one of the simplest classifiers there is, and works surprisingly well when classes form compact, roughly spherical clusters (which Iris mostly does).

### 3.1 &#9997;&#65039; Your turn: compute centroids

Complete `compute_centroids`: for each unique class label in `y_train`, compute the mean feature vector of the matching rows in `X_train`.

In [ ]:
def compute_centroids(X_train, y_train):
    """Return a dict mapping each class label to its centroid (mean feature vector)."""
    pass

### 3.2 &#9997;&#65039; Your turn: predict with centroids

Complete `predict_centroid`: for each row in `X`, compute its `euclidean_distance` to every centroid and predict the label of the closest one.

In [ ]:
def predict_centroid(X, centroids):
    """Predict a class label for every row in X using nearest-centroid."""
    pass

In [ ]:
centroids = compute_centroids(X_train, y_train)
centroid_preds = predict_centroid(X_test, centroids)

centroid_acc = accuracy_score(y_test, centroid_preds)
print(f"From-scratch centroid classifier accuracy: {centroid_acc:.3f}")

In [ ]:
sk_centroid = NearestCentroid()
sk_centroid.fit(X_train, y_train)
sk_centroid_acc = sk_centroid.score(X_test, y_test)
print(f"scikit-learn NearestCentroid accuracy:      {sk_centroid_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 5))
for label, name in enumerate(iris.target_names):
    subset = X_train[y_train == label]
    plt.scatter(subset[:, 2], subset[:, 3], alpha=0.5, label=name)
    c = centroids[label]
    plt.scatter(c[2], c[3], marker="X", s=200, edgecolor="black", linewidth=1.5)
plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.title("Class centroids (X) over training data")
plt.legend()
plt.show()

## 4. K-Nearest Neighbors Classifier (Iris)

**Idea:** to classify a new point, find the `k` closest points in the training set (by Euclidean distance) and take a majority vote of their labels. Unlike the centroid classifier, KNN makes no assumption that classes form single compact clusters — it can capture more complex decision boundaries, at the cost of comparing against every training point at prediction time.

### 4.1 &#9997;&#65039; Your turn: classify one point

Complete `knn_predict_one`:
1. Compute the distance from `x` to every row of `X_train`
2. Find the indices of the `k` smallest distances (`np.argsort` is handy here)
3. Return the most common label among those `k` neighbors (`collections.Counter` is handy here)

In [ ]:
def knn_predict_one(x, X_train, y_train, k):
    """Predict the class label for a single point x using k-nearest neighbors."""
    pass

`knn_predict` below just applies `knn_predict_one` to every row — no changes needed.

In [ ]:
def knn_predict(X, X_train, y_train, k):
    """Predict class labels for every row in X using k-nearest neighbors."""
    return np.array([knn_predict_one(x, X_train, y_train, k) for x in X])

In [ ]:
# Sanity check on a tiny hand-made dataset
mock_X_train = np.array([[0, 0], [0, 1], [5, 5], [5, 4]])
mock_y_train = np.array([0, 0, 1, 1])

assert knn_predict_one(np.array([0, 0.5]), mock_X_train, mock_y_train, k=1) == 0
assert knn_predict_one(np.array([5, 4.5]), mock_X_train, mock_y_train, k=3) == 1
print("Passed.")

In [ ]:
for k in [1, 3, 5]:
    knn_preds = knn_predict(X_test, X_train, y_train, k)
    print(f"k={k}: from-scratch KNN accuracy = {accuracy_score(y_test, knn_preds):.3f}")

In [ ]:
sk_knn = KNeighborsClassifier(n_neighbors=3)
sk_knn.fit(X_train, y_train)
print(f"scikit-learn KNeighborsClassifier (k=3) accuracy: {sk_knn.score(X_test, y_test):.3f}")

## 5. Naive Bayes Classifier (Play Tennis)

**Idea:** by Bayes' rule,

$$P(\text{class} \mid \text{features}) \propto P(\text{class}) \cdot \prod_i P(\text{feature}_i \mid \text{class})$$

The "naive" part is assuming every feature is conditionally independent given the class — rarely exactly true, but a surprisingly effective approximation, and it means we only ever need to estimate simple per-feature probabilities from counts.

We'll also use **Laplace (add-one) smoothing** so a feature value that never appeared with a given class in training doesn't zero out the whole probability: instead of `count / class_count`, we use `(count + 1) / (class_count + n_unique_values)`.

### 5.1 Priors

`compute_priors` estimates $P(\text{class})$ directly from the class frequencies — this one's done for you.

In [ ]:
def compute_priors(df, target_col):
    """Return a dict mapping each class label to P(class)."""
    counts = df[target_col].value_counts()
    return (counts / len(df)).to_dict()

### 5.2 &#9997;&#65039; Your turn: likelihoods

Complete `compute_likelihoods`. It should return a nested dict:
`likelihoods[class][feature][value] = P(value | class)`, using Laplace smoothing — for a given class and feature, `P(value | class) = (count_of_value_in_class + 1) / (class_count + n_unique_values_for_feature)`.

In [ ]:
def compute_likelihoods(df, target_col):
    """Return likelihoods[class][feature][value] = P(value | class), Laplace-smoothed."""
    pass

### 5.3 &#9997;&#65039; Your turn: predict

Complete `predict_naive_bayes`: for each class, multiply the prior by the likelihood of every observed feature value, then return the class with the highest score.

In [ ]:
def predict_naive_bayes(sample, priors, likelihoods):
    """sample: dict of {feature: value}. Returns the predicted class label."""
    pass

In [ ]:
priors = compute_priors(play_tennis, "PlayTennis")
likelihoods = compute_likelihoods(play_tennis, "PlayTennis")

new_day = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}
prediction = predict_naive_bayes(new_day, priors, likelihoods)
print(f"New day: {new_day}")
print(f"From-scratch prediction: {prediction}")

In [ ]:
encoder = OrdinalEncoder()
feature_cols = ["Outlook", "Temperature", "Humidity", "Wind"]
X_cat = encoder.fit_transform(play_tennis[feature_cols])
y_cat = play_tennis["PlayTennis"]

sk_nb = CategoricalNB()
sk_nb.fit(X_cat, y_cat)

new_day_encoded = encoder.transform(pd.DataFrame([new_day])[feature_cols])
sk_prediction = sk_nb.predict(new_day_encoded)[0]
print(f"scikit-learn CategoricalNB prediction:   {sk_prediction}")

## 6. Wrap-up

We covered:
- NumPy fundamentals: arrays, broadcasting, aggregations, and a from-scratch distance function
- **Nearest-Centroid** and **KNN** classifiers on the continuous-feature Iris dataset, built from scratch and verified against scikit-learn
- **Naive Bayes** on the categorical-feature Play Tennis dataset, built from scratch and verified against scikit-learn

Where to go next: try the centroid/KNN classifiers on all 4 Iris features instead of just petal length/width, experiment with different `k` values, or look at `sklearn.naive_bayes.GaussianNB` for Naive Bayes on continuous features.